# V15 – Netzwerktechnik: Praxis

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- typische **String-Methoden** (`.split`, `.startswith`, `.isdigit`, `.strip`) anwenden, um Netzwerkadressen und URLs zu zerlegen,
- eine **CSV-Datei** zeilenweise einlesen und die Anzahl der Datenzeilen bestimmen,
- eine **Port-Prüffunktion** bauen, die Gültigkeit über Zeichenart und Zahlenbereich abfragt.

## ⏱️ Zeitbudget
Ca. 30 Minuten.

## 🧭 TL;DR
Statt echte Netzwerk-Verbindungen aufzubauen, arbeiten wir **offline** mit vorgegebenen Strings und einer vorhandenen CSV-Datei. Das reicht, um die typischen Parser-Handgriffe für Netzwerkdaten einzuüben.

## 📦 Voraussetzungen
- [00_python_recap.ipynb](00_python_recap.ipynb)
- [01_theorie.ipynb](01_theorie.ipynb)

## 1. Eine Adresse zerlegen mit `.split`

Die Methode `.split(trennzeichen)` zerlegt einen String an jedem Auftreten des Trennzeichens und liefert eine Liste von Teilstrings. Für die Zerlegung von `host:port` ist genau das ideal. Ohne Argument wird an Whitespace getrennt.

In [1]:
adresse = "192.168.10.200:4840"
teile = adresse.split(":")
print(teile)
print("Host:", teile[0])
print("Port:", teile[1])

['192.168.10.200', '4840']
Host: 192.168.10.200
Port: 4840


### 1.1 Eine IP in vier Oktette zerlegen

Dieselbe Idee, nur diesmal mit Punkt als Trennzeichen. Danach wandeln wir jedes Teil-String per `int(...)` in eine Ganzzahl um.

In [2]:
ip = "10.0.5.42"
oktette = [int(x) for x in ip.split(".")]
print(oktette)
print("Erstes Oktett:", oktette[0])

[10, 0, 5, 42]
Erstes Oktett: 10


## 2. Präfixe erkennen mit `.startswith`

Um eine URL einzuordnen, willst du wissen, ob sie mit `http://` oder `https://` beginnt. `.startswith("http")` liefert einen Wahrheitswert zurück.

In [3]:
urls = ["https://beispiel.de", "opc.tcp://192.168.10.200:4840", "ftp://archiv"]
for url in urls:
    ist_web = url.startswith("http")
    print(f"{url:40s} → Web? {ist_web}")

https://beispiel.de                      → Web? True
opc.tcp://192.168.10.200:4840            → Web? False
ftp://archiv                             → Web? False


## 3. Ziffernprüfung mit `.isdigit`

Für einen Port-String brauchen wir Zweierlei: Er muss nur aus Ziffern bestehen, und die resultierende Zahl muss im Bereich 1..65535 liegen. Den ersten Teil erledigt `.isdigit()` in einem Schritt.

In [4]:
kandidaten = ["80", "4840", "abc", "70000", "-5", "0"]
for k in kandidaten:
    print(f"'{k}'.isdigit() = {k.isdigit()}")

'80'.isdigit() = True
'4840'.isdigit() = True
'abc'.isdigit() = False
'70000'.isdigit() = True
'-5'.isdigit() = False
'0'.isdigit() = True


> [!WARNING]
> `.isdigit()` ist bei negativen Zahlen `False`, weil das Minus-Zeichen keine Ziffer ist. Das ist in unserem Fall genau das gewünschte Verhalten (ein Port darf nicht negativ sein), kann in anderen Kontexten aber überraschen.

## 4. Weißraum entfernen mit `.strip`

Eingelesene Zeilen aus Dateien enden oft auf `\n` oder enthalten unschöne Leerzeichen am Rand. `.strip()` entfernt Whitespace am Anfang **und** Ende.

In [5]:
rohzeile = "  CNC-01  \n"
sauber = rohzeile.strip()
print(f"Roh:    '{rohzeile}'")
print(f"Sauber: '{sauber}'")

Roh:    '  CNC-01  
'
Sauber: 'CNC-01'


## 5. Fill-in-Blank 1 – Host und Port trennen

Zerlege die Adresse in Host (String) und Port (Integer). Nutze `.split` und `int(...)`.

In [6]:
adresse = "scada.halle.a:502"
host = ""    # TODO: richtigen Wert setzen
port = 0     # TODO: als int speichern

# ▶️ Selbstkontrolle
try:
    assert host == "scada.halle.a", f"host falsch: {host}"
    assert port == 502, f"port falsch: {port}"
    assert isinstance(port, int), "port muss ein int sein"
    print("✅ Host und Port korrekt zerlegt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ `host` oder `port` fehlt noch.")

❌ Noch nicht ganz: host falsch: 


## 6. Fill-in-Blank 2 – Protokoll aus URL ableiten

Setze `ist_sicher` auf `True`, wenn die URL mit `https` beginnt.

In [7]:
url = "https://mes.halle.a/api/status"
ist_sicher = False  # TODO: per startswith setzen

# ▶️ Selbstkontrolle
try:
    assert ist_sicher is True, "https-URL sollte als sicher erkannt werden"
    print("✅ Protokoll richtig erkannt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `ist_sicher` existiert noch nicht.")

❌ Noch nicht ganz: https-URL sollte als sicher erkannt werden


## 7. Fill-in-Blank 3 – Zeile säubern

Entferne Whitespace am Rand und prüfe, ob das Ergebnis exakt `"Modbus TCP"` lautet.

In [8]:
rohzeile = "   Modbus TCP\n"
sauber = rohzeile  # TODO: .strip() anwenden

# ▶️ Selbstkontrolle
try:
    assert sauber == "Modbus TCP", f"Nicht sauber: '{sauber}'"
    print("✅ Whitespace korrekt entfernt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `sauber` existiert noch nicht.")

❌ Noch nicht ganz: Nicht sauber: '   Modbus TCP
'


## 8. CSV-Datei zeilenweise einlesen

Eine CSV-Datei ist schlicht eine Textdatei mit einer Zeile pro Datensatz. Wir können sie mit `open()` und `with`-Block zeilenweise durchgehen. Die erste Zeile ist üblicherweise die Kopfzeile mit Spaltennamen – sie zählt nicht zu den Daten.

In [9]:
with open("netzwerk_pakete.csv", encoding="utf-8") as f:
    zeilen = f.readlines()

print(f"Gesamtanzahl Zeilen (inkl. Kopfzeile): {len(zeilen)}")
print(f"Anzahl Datensätze:                     {len(zeilen) - 1}")
print(f"Kopfzeile: {zeilen[0].strip()}")
print(f"Erster Datensatz: {zeilen[1].strip()}")

Gesamtanzahl Zeilen (inkl. Kopfzeile): 98
Anzahl Datensätze:                     97
Kopfzeile: Zeitstempel,Quell_IP,Ziel_IP,Protokoll,Port,Paketgroesse_Bytes,Maschine_ID
Erster Datensatz: 2024-01-15_08:00:12,192.168.10.100,192.168.10.200,TCP,502,156,CNC-01


### 8.1 Ein einzelnes Feld extrahieren

Jede Datenzeile lässt sich an `,` aufteilen. Laut Kopfzeile ist das fünfte Feld (Index 4) der **Port**. Hier ein Beispiel für die erste Datenzeile.

In [10]:
erste_daten = zeilen[1].strip()
felder = erste_daten.split(",")
print("Felder:", felder)
print("Port aus erster Zeile:", felder[4])

Felder: ['2024-01-15_08:00:12', '192.168.10.100', '192.168.10.200', 'TCP', '502', '156', 'CNC-01']
Port aus erster Zeile: 502


## 9. Mini-Challenge – `ist_valide_port`

Schreibe eine Funktion `ist_valide_port(port_str)`, die genau dann `True` zurückgibt, wenn

- der String `port_str` **nur Ziffern** enthält **und**
- die resultierende Zahl im Bereich `1 <= port <= 65535` liegt.

In allen anderen Fällen (leer, Buchstaben, zu groß, zu klein, Null) soll sie `False` zurückgeben.

In [11]:
def ist_valide_port(port_str):
    # TODO: Prüfung implementieren
    return False

# ▶️ Selbstkontrolle
try:
    assert ist_valide_port("80") is True
    assert ist_valide_port("65535") is True
    assert ist_valide_port("abc") is False
    assert ist_valide_port("70000") is False
    assert ist_valide_port("0") is False
    assert ist_valide_port("") is False
    print("✅ Challenge gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Funktion `ist_valide_port` existiert noch nicht.")

❌ Noch nicht ganz: 


## ✅ Zusammenfassung
- `.split`, `.startswith`, `.isdigit` und `.strip` decken den Großteil der String-Arbeit bei Netzwerkdaten ab.
- CSV-Dateien lassen sich mit `open()` und `readlines()` zeilenweise einlesen; die erste Zeile ist meist eine Kopfzeile.
- Gültige Ports lassen sich über eine Kombination aus Zeichenprüfung (`.isdigit`) und Bereichsprüfung (`1..65535`) bestimmen.

## ➡️ Nächster Schritt
Weiter mit [03_aufgaben.ipynb](03_aufgaben.ipynb).